# Sentiment Analysis based on gender

In this notebook, I will analyse some Airbnb customer reviews in some different languages. The main goal is to make a sentiment analysis and then look at the gender variable. There are more positive reviews from males or females?

In [19]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
import time
from textblob import TextBlob

### Sentiment Classification

The initial idea was to use Python VADER library for the comments' analysis. However, VADER works only with English and cannot manage multilingual input.  
So, I'll try to use an HuggingFace pretrained model based on mBERT transformer.

In [31]:
rdf = pd.read_csv("reviews.csv")
names = rdf["reviewer_name"]
rdf.columns

Index(['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments'], dtype='object')

In [11]:
rdf.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,23986,1175194,2012-04-24,1695229,Leah,We came to Milan for the Salone and had a fant...
1,23986,47872586,2015-09-21,17316381,Paolo,La zona è molto comoda e la via è tranquilla. ...
2,23986,70176179,2016-04-16,41686521,Naama,"Great apartment, clean and well equipped, grea..."
3,23986,90028316,2016-07-30,70469005,Morgane,L'appartement de Jérémy est idéal pour séjourn...
4,23986,96167583,2016-08-22,64015880,Zita,A first: Jeremy is an excellent and helpfull h...


In [33]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

cpu


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [48]:
def get_sentiment(texts, batch_size=8):
    results = []
    total = len(texts)
    start_time = time.time()
    
    for i in range(0, total, batch_size):
        
        batch_texts = texts[i:i+batch_size]
        
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        scores = torch.argmax(probs, dim=1)
        
        for score in scores:
            score = score.item()
            if score < 2:
                results.append('Negative')
            elif score > 2:
                results.append('Positive')
            else:
                results.append('Neutral')
    
    print(f"Completed! Total time: {time.time() - start_time:.1f} s")
    return results

In [45]:
get_sentiment(['oggi è una bella giornata', 'i am really sad'])

Processed 0/2 comments (0.0%)
Completed! Total time: 0.1 s


['Positive', 'Negative']

Unfortunately, mBERT models have a quite high computational weight, so for this little project, I will test the analysis on the first 5000 Airbnb comments.

In [49]:
rdf_5000 = rdf.head(5000)
comments = rdf_5000['comments'].tolist()
rdf_5000['sentiment'] = get_sentiment(comments)
rdf_5000.head()

Completed! Total time: 833.6 s


C:\Users\elena\AppData\Local\Temp\ipykernel_6616\2119696302.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rdf_5000['sentiment'] = get_sentiment(comments)


,listing_id,id,date,reviewer_id,reviewer_name,comments,sentiment
0,23986,1175194,2012-04-24,1695229,Leah,We came to Milan for the Salone and had a fant...,Positive
1,23986,47872586,2015-09-21,17316381,Paolo,La zona è molto comoda e la via è tranquilla. ...,Positive
2,23986,70176179,2016-04-16,41686521,Naama,"Great apartment, clean and well equipped, grea...",Positive
3,23986,90028316,2016-07-30,70469005,Morgane,L'appartement de Jérémy est idéal pour séjourn...,Positive
4,23986,96167583,2016-08-22,64015880,Zita,A first: Jeremy is an excellent and helpfull h...,Positive


In [50]:
rdf_5000['sentiment'].value_counts()

sentiment
Positive    4795
Neutral      145
Negative      60
Name: count, dtype: int64

The output class results quite imbalanced, we will remind it during the analysis.

### Gender Classification

Neither the gender class is included in the original dataframe, the only useful data to use is the reviewer's name. In this case I chose a different approach: I found online an exaustive name-gender dataset, and with a simple algorithm I classified the gender with good accuracy.

In [51]:
df = pd.read_csv("d1.csv")
df.shape

(4970296, 4)

In [52]:
df1 = df.drop(["code", "wgt"], axis=1)

In [53]:
male_count = 0
female_count = 0
notfound_count = 0
question_count = 0

gender_dict = dict(zip(df1['name'], df1['gender']))

def get_gender(name):
    name = name.lower()
    return gender_dict.get(name, None)

rdf_5000['gender'] = rdf_5000['reviewer_name'].apply(get_gender)

# Aggiungiamo i contatori
male_count = len(rdf_5000[rdf_5000['gender'] == 'M'])
female_count = len(rdf_5000[rdf_5000['gender'] == 'F'])
notfound_count = len(rdf_5000[rdf_5000['gender'].isna()])
question_count = len(rdf_5000[rdf_5000['gender'].notna() & ~rdf_5000['gender'].isin(['M', 'F'])])

print(rdf_5000.head())
print(f"Males: {male_count}")
print(f"Females: {female_count}")
print(f"Not Found: {notfound_count}")
print(f"Unknown Gender(?): {question_count}")


   listing_id        id        date  reviewer_id reviewer_name  \
0       23986   1175194  2012-04-24      1695229          Leah   
1       23986  47872586  2015-09-21     17316381         Paolo   
2       23986  70176179  2016-04-16     41686521         Naama   
3       23986  90028316  2016-07-30     70469005       Morgane   
4       23986  96167583  2016-08-22     64015880          Zita   

                                            comments sentiment gender  
0  We came to Milan for the Salone and had a fant...  Positive      F  
1  La zona è molto comoda e la via è tranquilla. ...  Positive      M  
2  Great apartment, clean and well equipped, grea...  Positive      F  
3  L'appartement de Jérémy est idéal pour séjourn...  Positive      F  
4  A first: Jeremy is an excellent and helpfull h...  Positive      F  
Males: 2719
Females: 1926
Not Found: 327
Unknown Gender(?): 28


C:\Users\elena\AppData\Local\Temp\ipykernel_6616\3036787662.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rdf_5000['gender'] = rdf_5000['reviewer_name'].apply(get_gender)
